# Scenario 3 - LLM XAI and LLM-as-Judge


In [7]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load credentials from .env file
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not OPENROUTER_API_KEY or not GROQ_API_KEY:
    raise ValueError("Missing API Keys in .env. Please check OPENROUTER_API_KEY and GROQ_API_KEY.")

# Initialize API Clients
client_openrouter = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

client_groq = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY,
)

# Your 3 Competitor Models
MODEL_CONFIG = {
    "Gemma-4-31B": {
        "model_id": "google/gemma-4-31b-it:free", 
        "client": client_openrouter,
        "is_openrouter": True
    },
    # "Qwen-3-32B": {
    #     "model_id": "qwen/qwen3-32b",
    #     "client": client_groq,
    #     "is_openrouter": False
    # },
    # "Llama-3.3-70B": {
    #     "model_id": "llama-3.3-70b-versatile",
    #     "client": client_groq,
    #     "is_openrouter": False
    # }
}

print("Generator models initialized. Ready to generate data.")

Generator models initialized. Ready to generate data.


## 1. Setup API Client dan Konfigurasi Model


In [2]:
import pandas as pd
from bertopic import BERTopic

# 1. Identify the winning arm from Skenario 2
winner_meta_path = r"skenario2/output/scenario2_winner_for_scenario3.csv"
df_winner = pd.read_csv(winner_meta_path)
winning_arm = df_winner.iloc[0]['arm']
print(f"Skenario 2 Winning Arm: {winning_arm}")

# 2. Load the trained BERTopic model
model_path = r"skenario2/output/scenario2_best_model.pkl"
print("Loading BERTopic model (this may take a moment)...")
topic_model = BERTopic.load(model_path)

# 3. Extract the full topic info dataframe (which contains Representative_Docs)
df_topic_info = topic_model.get_topic_info()

# Filter out outlier topic (-1)
df_topic_info = df_topic_info[df_topic_info['Topic'] != -1]

print(f"Extracted {len(df_topic_info)} topics from the model.")
df_topic_info[['Topic', 'Count', 'Name']].head()


Skenario 2 Winning Arm: HDBSCAN_reduced_outliers
Loading BERTopic model (this may take a moment)...
Extracted 25 topics from the model.


,Topic,Count,Name
0,0,2783,0_skin_dermatology_psoriasis_treatment
1,1,2396,1_urology_prostate_urinary_bladder
2,2,2410,2_mental_psychiatric_health_mental health
3,3,2358,3_cardiology_risk_heart_cardiovascular
4,4,3222,4_immune_cells_cell_tumor


## 2. Load Winner Skenario 2 dan Topic Info


In [3]:
import time
import json

def build_detailed_xai_prompt(keywords, docs):
    """Generates a detailed prompt demanding deep clinical reasoning from the LLM."""
    docs_block = "\n\n".join([f"Abstract {i+1}:\n{doc}" for i, doc in enumerate(docs[:3])])
    
    prompt = f"""You are an expert clinical taxonomist and AI system specialized in explainable clinical NLP.
Analyze the following topic cluster derived from medical abstracts.

--- TOP WORD REPRESENTATION (c-TF-IDF) ---
{", ".join(keywords)}

--- REPRESENTATIVE CLINICAL ABSTRACTS ---
{docs_block}

---
Your task is to generate a comprehensive, highly detailed taxonomic description of this topic cluster.
You must return your output strictly in the following JSON format:
{{
  "refined_label": "A concise, professional medical/scientific title (2-4 words)",
  "scientific_explanation": "A detailed 3-4 sentence explanation of the overarching medical field, specific pathology, or clinical intervention described.",
  "key_semantic_themes": [
    "Theme 1 (e.g., specific clinical outcomes analyzed)",
    "Theme 2 (e.g., patient demographics or therapeutic interventions identified)"
  ],
  "label_reasoning": "Explain the step-by-step reasoning behind choosing this specific label. Point out which keywords or sentences in the abstracts directly dictated your decision, and why other generic labels were rejected.",
  "cohesion_score": 8,
  "cohesion_reasoning": "A detailed explanation of why you gave this cohesion score. Mention if there are outlier concepts or if the abstracts align perfectly."
}}"""
    return prompt

def query_model_with_retry(model_name, prompt, max_retries=3, delay=15):
    """Queries OpenRouter or Groq with safe checks for empty choices arrays."""
    config = MODEL_CONFIG.get(model_name)
    if not config:
        print(f"  [Error] Model {model_name} is not defined in MODEL_CONFIG.")
        return None
        
    client = config["client"]
    model_id = config["model_id"]
    is_openrouter = config.get("is_openrouter", False)
    
    for attempt in range(max_retries):
        try:
            if is_openrouter:
                response = client.chat.completions.create(
                    model=model_id,
                    messages=[
                        {"role": "system", "content": "You are a precise clinical assistant. You always respond only with valid, parsable JSON matching the requested schema."},
                        {"role": "user", "content": prompt}
                    ],
                    extra_body={
                        "safety_settings": [
                            {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
                            {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
                            {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
                            {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
                            {"category": "HARM_CATEGORY_CIVIC_INTEGRITY", "threshold": "BLOCK_NONE"}
                        ]
                    },
                    temperature=0.1,
                    max_tokens=1024,
                    response_format={"type": "json_object"},
                    extra_headers={
                        "HTTP-Referer": "https://github.com/Inventiner/text-mining-project",
                        "X-Title": "PubMed Topic Modeling XAI",
                    }
                )
            else:
                response = client.chat.completions.create(
                    model=model_id,
                    messages=[
                        {"role": "system", "content": "You are a precise clinical assistant. You always respond only with valid, parsable JSON matching the requested schema."},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0.1,
                    max_tokens=1024,
                    response_format={"type": "json_object"}
                )
            
            # Safe validation check
            if response and hasattr(response, 'choices') and len(response.choices) > 0:
                choice = response.choices[0]
                if hasattr(choice, 'message') and choice.message and choice.message.content:
                    return choice.message.content
            
            print(f"  [Warning] {model_name} returned an empty response.")
            return None
            
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower() or "limit" in str(e).lower():
                print(f"  [Rate Limit] {model_name} busy. Retrying in {delay}s... (Attempt {attempt+1}/{max_retries})")
                time.sleep(delay)
            else:
                print(f"  [Error] {model_name} execution failed: {e}")
                break
    return None

## 3. Prompt dan Helper Query LLM


In [ ]:
import os
import json
import ast
import pandas as pd

CSV_OUTPUT_PATH = r"xai/s3_xai_detailed_comparison_hdbscan_25.csv"

# Specify exactly which topic IDs to run/re-run (e.g., [0, 8, 11])
TARGET_TOPIC_IDS = [0, 8, 11]  

# Standardized Column Layout
REQUIRED_COLUMNS = [
    "topic_id", "doc_count", "original_name", "top_keywords",
    
    # Gemma-4-31B-it Columns
    "Gemma-4-31B-it_status", "Gemma-4-31B-it_label", "Gemma-4-31B-it_explanation", 
    "Gemma-4-31B-it_themes", "Gemma-4-31B-it_reasoning", "Gemma-4-31B-it_cohesion", 
    "Gemma-4-31B-it_cohesion_reasoning", "Gemma-4-31B-it_raw_error",
    
    # Qwen-3-32B Columns
    "Qwen-3-32B_status", "Qwen-3-32B_label", "Qwen-3-32B_explanation", 
    "Qwen-3-32B_themes", "Qwen-3-32B_reasoning", "Qwen-3-32B_cohesion", 
    "Qwen-3-32B_cohesion_reasoning", "Qwen-3-32B_raw_error",
    
    # Llama-3.3-70B Columns
    "Llama-3.3-70B_status", "Llama-3.3-70B_label", "Llama-3.3-70B_explanation", 
    "Llama-3.3-70B_themes", "Llama-3.3-70B_reasoning", "Llama-3.3-70B_cohesion", 
    "Llama-3.3-70B_cohesion_reasoning", "Llama-3.3-70B_raw_error"
]

def safe_get_list(val):
    """Safely handles both native Python lists and stringified lists from CSVs."""
    if isinstance(val, list):
        return val
    if pd.isna(val):
        return []
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError, TypeError):
        return [item.strip().strip("'\"") for item in str(val).strip("[]").split(",")]

# Load input data
if 'df_topic_info' in locals():
    df_input = df_topic_info.copy()
else:
    df_input = pd.read_csv(r"skenario2/output/scenario2_topics_hdbscan_reduced_outliers.csv")

df_input = df_input[df_input['Topic'] != -1]

# Load existing outputs to preserve successful rows
if os.path.exists(CSV_OUTPUT_PATH):
    try:
        df_existing = pd.read_csv(CSV_OUTPUT_PATH)
        processed_topics = df_existing.to_dict('records')
        print(f"Loaded existing file. Preserving other columns while updating: {TARGET_TOPIC_IDS}")
    except Exception as e:
        print(f"Starting fresh: {e}")
        processed_topics = []
else:
    processed_topics = []

# Loop ONLY through the target IDs
for topic_id in TARGET_TOPIC_IDS:
    source_row = df_input[df_input['Topic'] == topic_id]
    if source_row.empty:
        print(f"Topic {topic_id} not found in input data. Skipping.")
        continue
        
    row = source_row.iloc[0]
    count_docs = int(row['Count'])
    original_name = row['Name']
    
    keywords = safe_get_list(row['Representation'])
    docs = safe_get_list(row['Representative_Docs'])
    
    print("="*80)
    print(f"PROCESSING TARGET TOPIC {topic_id} ({count_docs} docs) - Original: {original_name}")
    print("="*80)
    
    prompt = build_detailed_xai_prompt(keywords, docs)
    
    # 1. Look for existing row in database to preserve other models' data
    row_result = next((item for item in processed_topics if item["topic_id"] == topic_id), None)
    
    if not row_result:
        # Create a fresh record dictionary if it doesn't exist
        row_result = {col: None for col in REQUIRED_COLUMNS}
        row_result.update({
            "topic_id": topic_id,
            "doc_count": count_docs,
            "original_name": original_name,
            "top_keywords": ", ".join(keywords[:5])
        })
    else:
        # Convert any NaN values from pandas load to None for clean updates
        row_result = {k: (None if pd.isna(v) else v) for k, v in row_result.items()}
    
    # 2. Reset values ONLY for the models defined in Cell 1 MODEL_CONFIG
    for model_name in MODEL_CONFIG.keys():
        row_result[f"{model_name}_label"] = None
        row_result[f"{model_name}_explanation"] = None
        row_result[f"{model_name}_themes"] = None
        row_result[f"{model_name}_reasoning"] = None
        row_result[f"{model_name}_cohesion"] = None
        row_result[f"{model_name}_cohesion_reasoning"] = None
        row_result[f"{model_name}_status"] = "PENDING"
        row_result[f"{model_name}_raw_error"] = None

    # 3. Query only the active models
    for model_name in MODEL_CONFIG.keys():
        print(f"\n---> Querying {model_name}...")
        raw_json = query_model_with_retry(model_name, prompt)
        
        if raw_json:
            try:
                parsed = json.loads(raw_json)
                
                # Assign results
                row_result[f"{model_name}_label"] = parsed.get("refined_label")
                row_result[f"{model_name}_explanation"] = parsed.get("scientific_explanation")
                row_result[f"{model_name}_themes"] = ", ".join(parsed.get("key_semantic_themes", []))
                row_result[f"{model_name}_reasoning"] = parsed.get("label_reasoning")
                row_result[f"{model_name}_cohesion"] = parsed.get("cohesion_score")
                row_result[f"{model_name}_cohesion_reasoning"] = parsed.get("cohesion_reasoning")
                row_result[f"{model_name}_status"] = "SUCCESS"
                row_result[f"{model_name}_raw_error"] = None
                
                # Live print
                print(f"\n[{model_name} Live Debug Output]:")
                print(f"  Refined Label : {parsed.get('refined_label')}")
                print(f"  Cohesion Score: {parsed.get('cohesion_score')}")
                print("-" * 50)
                
            except json.JSONDecodeError:
                print(f"  [Error] Parsing error with {model_name}'s JSON structure.")
                row_result[f"{model_name}_raw_error"] = raw_json
                row_result[f"{model_name}_status"] = "JSON_DECODE_ERROR"
        else:
            row_result[f"{model_name}_status"] = "API_ERROR"
            
    # Update our storage list securely without touching columns of inactive models
    processed_topics = [item for item in processed_topics if item["topic_id"] != topic_id]
    processed_topics.append(row_result)
    
    # Save incrementally
    df_temp = pd.DataFrame(processed_topics)
    df_temp = df_temp.reindex(columns=REQUIRED_COLUMNS)
    df_temp = df_temp.sort_values(by="topic_id")
    df_temp.to_csv(CSV_OUTPUT_PATH, index=False)

print("\n" + "="*80)
print(f"Safe run complete. Preserved other data and successfully updated: {CSV_OUTPUT_PATH}")
print("="*80)

Loaded existing file. Target topics to be updated: [13]
PROCESSING TARGET TOPIC 13 (448 docs) - Original: 13_bone_regeneration_tissue_repair

---> Querying Gemma-4-31B...

[Gemma-4-31B Live Debug Output]:
  Refined Label : Bone Tissue Engineering Scaffolds
  Cohesion Score: 9
--------------------------------------------------

Targeted 3-Model run complete. Results saved to: xai/s3_xai_detailed_comparison_hdbscan_25(llm-as-judge).csv


## 4. Generate XAI per Topic


In [22]:
import os
import json
import ast
import pandas as pd
import time

# 1. Paths Configuration
XAI_INPUT_PATH = r"xai/s3_xai_detailed_comparison_hdbscan_25(llm-as-judge).csv"
JUDGE_OUTPUT_PATH = r"xai/s3_xai_judged_results_hdbscan_25.csv"

# Specify exactly which topic IDs to judge or re-judge (e.g. list(range(25)))
TARGET_TOPIC_IDS = [2,3,8,9,10,15,21,23,24]

# 2. Judge Model Configuration
JUDGE_CONFIG = {
    "model_id": "google/gemini-3.1-flash-lite",
    "client": client_openrouter
}

# 3. Blind Mapping Definition
ANONYMOUS_MAPPING = {
    "Model_A": "Gemma-4-31B",
    "Model_B": "Qwen-3-32B",
    "Model_C": "Llama-3.3-70B"
}

# Enforced Standard Column Order (Unpacked for easy numerical analysis)
JUDGE_COLUMNS = [
    "topic_id", "original_name", "top_keywords",
    
    # Gemma-4-31B Judging Columns
    "Gemma-4-31B_faithfulness", "Gemma-4-31B_specificity", 
    "Gemma-4-31B_interpretability", "Gemma-4-31B_linguistic_utility", 
    "Gemma-4-31B_total", "Gemma-4-31B_justification_detail",
    
    # Qwen-3-32B Judging Columns
    "Qwen-3-32B_faithfulness", "Qwen-3-32B_specificity", 
    "Qwen-3-32B_interpretability", "Qwen-3-32B_linguistic_utility", 
    "Qwen-3-32B_total", "Qwen-3-32B_justification_detail",
    
    # Llama-3.3-70B Judging Columns
    "Llama-3.3-70B_faithfulness", "Llama-3.3-70B_specificity", 
    "Llama-3.3-70B_interpretability", "Llama-3.3-70B_linguistic_utility", 
    "Llama-3.3-70B_total", "Llama-3.3-70B_justification_detail",
    
    "winner_model", "judgment_reasoning"
]

def safe_get_list(val):
    """Safely handles both native Python lists and stringified lists from CSVs."""
    if isinstance(val, list):
        return val
    if pd.isna(val):
        return []
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError, TypeError):
        return [item.strip().strip("'\"") for item in str(val).strip("[]").split(",")]

def build_judge_prompt(keywords, docs, model_a_data, model_b_data, model_c_data):
    """Constructs a blind comparative evaluation prompt using a 4-criteria rubric."""
    docs_block = "\n\n".join([f"Abstract {i+1}:\n{doc}" for i, doc in enumerate(docs[:3])])
    
    prompt = f"""You are an independent, expert clinical NLP referee. Your task is to evaluate and rank the quality of three candidate topic explanations generated by different AI models.

--- INPUT DATA FOR THE TOPIC ---
Top Keywords: {keywords}

Representative Abstracts: 
{docs_block}

--- CANDIDATES TO EVALUATE (BLIND MATCHUP) ---
[Model_A]
Refined Label: {model_a_data['label']}
Explanation: {model_a_data['explanation']}
Themes: {model_a_data['themes']}
Reasoning: {model_a_data['reasoning']}

[Model_B]
Refined Label: {model_b_data['label']}
Explanation: {model_b_data['explanation']}
Themes: {model_b_data['themes']}
Reasoning: {model_b_data['reasoning']}

[Model_C]
Refined Label: {model_c_data['label']}
Explanation: {model_c_data['explanation']}
Themes: {model_c_data['themes']}
Reasoning: {model_c_data['reasoning']}

--- CALIBRATED EVALUATION RUBRIC ---
Evaluate each candidate on a scale of 1 to 5 for each of the 4 metrics below.
CRITICAL SCORING INSTRUCTION: Do not award scores of 1 or 5 unless performance is exceptionally terrible or flawless. 
- Map high-quality outputs to a score of 4.
- Map poor-quality outputs to a score of 2.
- Map baseline, average outputs to a score of 3.

METRICS:
1. FAITHFULNESS (1-5): Is the explanation grounded in the provided abstracts? Deduct points for external medical concepts or drug names not present in the source text.
2. CLINICAL SPECIFICITY (1-5): Does the label capture the precise medical nuance of this specific cluster, or is it too broad/generic?
3. INTERPRETABILITY (1-5): Does the explanation logically outline why the keywords and abstracts are connected, making it easy for a human researcher to understand the cluster's context?
4. LINGUISTIC UTILITY (1-5): Is the explanation concise, clearly structured, and free from repetitive or bloated sentences?

Output your decision strictly in this JSON format:
{{
  "scores": {{
    "Model_A": {{
      "faithfulness": {{"score": 3, "reason": "Justification for faithfulness"}},
      "specificity": {{"score": 4, "reason": "Justification for specificity"}},
      "interpretability": {{"score": 3, "reason": "Justification for interpretability"}},
      "linguistic_utility": {{"score": 3, "reason": "Justification for utility"}},
      "total": 13
    }},
    "Model_B": {{
      "faithfulness": {{"score": 4, "reason": "Justification for faithfulness"}},
      "specificity": {{"score": 3, "reason": "Justification for specificity"}},
      "interpretability": {{"score": 4, "reason": "Justification for interpretability"}},
      "linguistic_utility": {{"score": 4, "reason": "Justification for utility"}},
      "total": 15
    }},
    "Model_C": {{
      "faithfulness": {{"score": 3, "reason": "Justification for faithfulness"}},
      "specificity": {{"score": 3, "reason": "Justification for specificity"}},
      "interpretability": {{"score": 3, "reason": "Justification for interpretability"}},
      "linguistic_utility": {{"score": 3, "reason": "Justification for utility"}},
      "total": 12
    }}
  }},
  "winner": "Model_B", // Must be exactly "Model_A", "Model_B", or "Model_C"
  "overall_reasoning": "A brief overall comparison of the clinical precision and interpretability of the winner against the other candidates."
}}"""
    return prompt

def query_judge_with_retry(prompt, max_retries=3, delay=15):
    """Sends prompt to the Gemini Judge via OpenRouter, handling rate limits."""
    client = JUDGE_CONFIG["client"]
    model_id = JUDGE_CONFIG["model_id"]
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": "You are a precise clinical referee. You always respond only with valid, parsable JSON matching the requested schema."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,
                max_tokens=2500,  # Increased headroom for detailed reasons
                response_format={"type": "json_object"},
                extra_headers={
                    "HTTP-Referer": "https://github.com/your-username/text-mining-project",
                    "X-Title": "PubMed Topic Modeling XAI Judge",
                }
            )
            
            if response and hasattr(response, 'choices') and len(response.choices) > 0:
                choice = response.choices[0]
                if hasattr(choice, 'message') and choice.message and choice.message.content:
                    return choice.message.content
            print("  [Warning] Judge returned empty response.")
            return None
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower() or "limit" in str(e).lower():
                print(f"  [Rate Limit] Judge busy. Retrying in {delay}s... (Attempt {attempt+1}/{max_retries})")
                time.sleep(delay)
            else:
                print(f"  [Error] Judge API call failed: {e}")
                break
    return None

# Load competitor run outputs
if not os.path.exists(XAI_INPUT_PATH):
    raise FileNotFoundError(f"Competitor data not found at: {XAI_INPUT_PATH}.")

df_xai = pd.read_csv(XAI_INPUT_PATH)
df_input = pd.read_csv(r"skenario2/output/scenario2_topics_hdbscan_reduced_outliers.csv") if 'df_topic_info' not in locals() else df_topic_info.copy()
df_input = df_input[df_input['Topic'] != -1]

# Load existing progress to preserve other rows
if os.path.exists(JUDGE_OUTPUT_PATH):
    try:
        df_existing_judge = pd.read_csv(JUDGE_OUTPUT_PATH)
        judge_records = df_existing_judge.to_dict('records')
        print(f"Loaded existing judged progress. Will update only topics: {TARGET_TOPIC_IDS}")
    except Exception as e:
        print(f"Starting fresh: {e}")
        judge_records = []
else:
    judge_records = []

# Loop ONLY through specified TARGET_TOPIC_IDS
for topic_id in TARGET_TOPIC_IDS:
    xai_row = df_xai[df_xai['topic_id'] == topic_id]
    if xai_row.empty:
        print(f"Topic {topic_id} not found in competitor data file. Skipping.")
        continue
    
    row = xai_row.iloc[0]
    original_name = row['original_name']
    keywords = row['top_keywords']
    
    # Verify that we have successful generations from all 3 competitors
    has_gemma = pd.notna(row.get('Gemma-4-31B_label'))
    has_qwen = pd.notna(row.get('Qwen-3-32B_label'))
    has_llama = pd.notna(row.get('Llama-3.3-70B_label'))
    
    if not (has_gemma and has_qwen and has_llama):
        print(f"Skipping Topic {topic_id}: One or more competitor labels are missing.")
        continue
        
    # Find raw abstracts from input data
    src_row = df_input[df_input['Topic'] == topic_id]
    if src_row.empty:
        print(f"Source abstracts for Topic {topic_id} not found in input data. Skipping.")
        continue
    docs = safe_get_list(src_row.iloc[0]['Representative_Docs'])
    
    print("="*80)
    print(f"JUDGING TARGET TOPIC {topic_id} - Original: {original_name}")
    print("="*80)
    
    # Anonymize outputs mapping correctly
    model_a_data = {
        "label": row['Gemma-4-31B_label'],
        "explanation": row['Gemma-4-31B_explanation'],
        "themes": row['Gemma-4-31B_themes'],
        "reasoning": row['Gemma-4-31B_reasoning']
    }
    model_b_data = {
        "label": row['Qwen-3-32B_label'],
        "explanation": row['Qwen-3-32B_explanation'],
        "themes": row['Qwen-3-32B_themes'],
        "reasoning": row['Qwen-3-32B_reasoning']
    }
    model_c_data = {
        "label": row['Llama-3.3-70B_label'],
        "explanation": row['Llama-3.3-70B_explanation'],
        "themes": row['Llama-3.3-70B_themes'],
        "reasoning": row['Llama-3.3-70B_reasoning']
    }
    
    prompt = build_judge_prompt(keywords, docs, model_a_data, model_b_data, model_c_data)
    
    raw_json = query_judge_with_retry(prompt)
    if raw_json:
        try:
            parsed = json.loads(raw_json)
            
            scores = parsed.get("scores", {})
            anonymous_winner = parsed.get("winner")
            actual_winner = ANONYMOUS_MAPPING.get(anonymous_winner, "UNKNOWN")
            
            # Construct mapped record for this topic
            record = {
                "topic_id": topic_id,
                "original_name": original_name,
                "top_keywords": keywords,
                "winner_model": actual_winner,
                "judgment_reasoning": parsed.get("overall_reasoning")
            }
            
            # Dynamically map Model A/B/C sub-scores and textual reasons to the actual models
            model_key_mapping = {
                "Model_A": "Gemma-4-31B",
                "Model_B": "Qwen-3-32B",
                "Model_C": "Llama-3.3-70B"
            }
            
            for anon_id, real_name in model_key_mapping.items():
                m_score = scores.get(anon_id, {})
                
                record[f"{real_name}_faithfulness"] = m_score.get("faithfulness", {}).get("score")
                record[f"{real_name}_specificity"] = m_score.get("specificity", {}).get("score")
                record[f"{real_name}_interpretability"] = m_score.get("interpretability", {}).get("score")
                record[f"{real_name}_linguistic_utility"] = m_score.get("linguistic_utility", {}).get("score")
                record[f"{real_name}_total"] = m_score.get("total")
                
                # Consolidate raw text justifications in a single text column to avoid wide tables
                m_justification = (
                    f"Faithfulness: {m_score.get('faithfulness', {}).get('reason')} | "
                    f"Specificity: {m_score.get('specificity', {}).get('reason')} | "
                    f"Interpretability: {m_score.get('interpretability', {}).get('reason')} | "
                    f"Utility: {m_score.get('linguistic_utility', {}).get('reason')}"
                )
                record[f"{real_name}_justification_detail"] = m_justification

            # Live Print
            print(f"\n[Judge decision]:")
            print(f"  Gemma Score : {record['Gemma-4-31B_total']}/20")
            print(f"  Qwen Score  : {record['Qwen-3-32B_total']}/20")
            print(f"  Llama Score : {record['Llama-3.3-70B_total']}/20")
            print(f"  Winner      : {actual_winner}")
            print(f"  Reasoning   : {parsed.get('overall_reasoning')}")
            print("-" * 50)
            
            # Overwrite old record if it exists
            judge_records = [item for item in judge_records if item["topic_id"] != topic_id]
            judge_records.append(record)
            
            # Save progress incrementally
            df_temp = pd.DataFrame(judge_records)
            df_temp = df_temp.reindex(columns=JUDGE_COLUMNS)
            df_temp = df_temp.sort_values(by="topic_id")
            df_temp.to_csv(JUDGE_OUTPUT_PATH, index=False)
            
        except json.JSONDecodeError:
            print("  [Error] Judge response was not in valid JSON format.")
            print(f"  Raw response: {raw_json}")
    else:
        print("  [Error] Judge API request failed.")

print("\n" + "="*80)
print(f"Targeted evaluation complete. Judged results saved to: {JUDGE_OUTPUT_PATH}")
print("="*80)

Loaded existing judged progress. Will update only topics: [2, 3, 8, 9, 10, 15, 21, 23, 24]
JUDGING TARGET TOPIC 2 - Original: 2_mental_psychiatric_health_mental health

[Judge decision]:
  Gemma Score : 16/20
  Qwen Score  : 15/20
  Llama Score : 11/20
  Winner      : Gemma-4-31B
  Reasoning   : Model_A provided the most accurate and nuanced label, 'Mental Health Systems Integration,' which perfectly encapsulates the common theme of implementation and systemic challenges across all three abstracts. Model_C struggled by over-emphasizing 'precision medicine,' which is only a major theme in one of the three abstracts, making its label and explanation less representative of the entire cluster.
--------------------------------------------------
JUDGING TARGET TOPIC 3 - Original: 3_cardiology_risk_heart_cardiovascular

[Judge decision]:
  Gemma Score : 16/20
  Qwen Score  : 16/20
  Llama Score : 12/20
  Winner      : Gemma-4-31B
  Reasoning   : Model A is the winner because it identifies the

## 5. LLM-as-Judge dan Penyimpanan Hasil
